# M4 — Backtesting

Traduce las predicciones en una estrategia larga/liquidez, la simula con
`Backtesting.py` (`FractionalBacktest`) sobre el test de 2025 y la compara
con Buy & Hold.

Corresponde a la Sección *Módulo 4* del Capítulo 4 (OE4, OE5).
El ratio de Sharpe usa `R_f = 0`, la misma convención que la simulación,
en la que la liquidez no renta.

In [ ]:
import pandas as pd
from tfm import backtest, data, features, validation
from tfm.models import build_model_specs

raw = data.load_raw()
ticker, model_name = "AAPL", "xgboost"

dataset = features.build_dataset(raw[ticker])
train, test = validation.temporal_split(dataset)
cols = features.feature_columns(dataset)
spec = build_model_specs()[model_name]
model, _ = validation.tune_model(spec, train[cols], train["target"])
preds = pd.Series(model.predict(test[cols]), index=test.index)

In [ ]:
prices_test = raw[ticker].loc[test.index.min():]
strategy = backtest.run_strategy_backtest(prices_test, preds, ticker)
bh = backtest.run_buy_and_hold(prices_test, ticker)

print("Strategy :", strategy["metrics"])
print("Buy&Hold :", bh["metrics"])
backtest.compare_vs_benchmark(strategy, bh)